# _Retrieval-Augmented Generation_ (RAG)

A Recuperação Aumentada por Geração (RAG) é uma arquitetura de sistemas de inteligência artificial que combina um LLM com um mecanismo de recuperação de informações em fontes de dados externas.

Formalmente, o RAG opera através de um processo de três etapas principais:

- **Recuperação (_Retrieval_):** a partir de uma consulta (_query_) do usuário, o sistema utiliza métricas de similaridade (como a similaridade por cosseno) para identificar e extrair fragmentos de documentos relevantes de uma base de conhecimento externa previamente indexada.
  - Os fragmentos de texto (_chunks_) são processados por meio de modelos de _Embeddings_, que são redes neurais responsáveis por mapear cadeias de texto em vetores numéricos de alta dimensão em um espaço vetorial contínuo. Esse processo, que é chamado de vetorização, transforma cada pedaço de texto em um vetor denso (uma lista de números decimais que representa as características semânticas daquele conteúdo), garantindo que sentenças com significados semânticos próximos ocupem posições próximas nesse espaço vetorial.
  - Não só o texto puro dos fragmentos é armazenado, mas cada fragmentos é associado ao seu respectivo vetor de _embeddings_ e metadados.
  - Quando uma consulta (_query_) é feita pelo usuário, ela também é convertida em um vetor pelo mesmo modelo de _embeddings_. O sistema, então, faz uma **busca por similaridade**, calculando a distância (geralmente a similaridade de cosseno) entre o vetor da consulta e todos os vetores armazenados no banco. Os fragmentos cujos vetores apresentam a maior proximidade matemática com o vetor da consulta são selecionados como o contexto mais relevante.
- **Contextualização (_Augmentation_):** os fragmentos recuperados são concatenados e inseridos no _prompt_ que será enviado ao LLM, servindo como base factual e contexto restritivo para a resposta.
- **Geração (_Generation_):** O modelo de linguagem processa a consulta do usuário integrada ao contexto fornecido e gera uma resposta baseada estritamente nas informações contidas nos documentos recuperados, reduzindo a dependência do conhecimento paramétrico do modelo e mitigando alucinações.

Dessa forma, o sistema não busca por correspondência exata de palavras-chave, mas por proximidade de significado, o que permite que a recuperação seja muito mais precisa em relação à intenção do usuário do que métodos tradicionais de busca.

O foco central desta técnica é fornecer aos modelos de linguagem acesso a dados atualizados ou privados, garantindo que a geração de texto seja fundamentada em evidências documentais concretas fornecidas em tempo de execução.

## Recupera conteúdo de documento externo - Arquivo `.txt`

Vamos começar defindo o objeto do modelo de linguagem:

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
api_key       = os.getenv("OPENAI_API_KEY")
model_version = os.getenv("OPENAI_MODEL_VERSION")

llm = ChatOpenAI(
    model=model_version,
    api_key=api_key
)

A primeira coisa a ser feita é ler o conteúdo de um documento, faremos isso utilizando a classe [`TextLoader`](https://reference.langchain.com/python/langchain-community/document_loaders/text/TextLoader) para ler o conteúdo de um documento do tipo `.txt`:

In [2]:
from langchain_community.document_loaders import TextLoader

# carrega conteudo do documento
text_document = TextLoader(file_path="../documents/GTB_gold_Nov23.txt", encoding="utf-8").load()
text_document

/tmp/ipykernel_151693/3869960064.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../documents/GTB_gold_Nov23.txt'}, page_content='\n1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  para  a \nM

## Quebrando conteúdo do documento em pedaços

Agora que temos o conteúdo do documento, podemos dividí-lo em pedaços (_chunks_). Faremos isso através da classe [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter), que quebra o texto de forma recursiva, tentando achar o caracter que melhor funciona. O texto não é quebrado de forma puramente matemática, mas sim de forma recursiva, tentando dividir o texto respeitando a estrutura natural da escrita, buscando quebras de parágrafo, depois quebras de linha e, finalmente, espaços entre palavras. Isso aumenta a probabilidade de que cada _chunk_ contenha uma informação coesa e completa.

Para isso, devemos informar o tamanho máximo dos _chunks_ (`chunk_size`) e também o tamanho da sobreposição (`chunk_overlap`). Quando um novo _chunk_ é definido, é adicionado um bloco do tamanho da sobreposição vindo do _chunk_ anterior. Essa prática é adotada para evitar que o sentido de uma frase ou parágrafo seja perdido se o corte de caracteres ocorrer exatamente no meio de uma ideia central. A sobreposição mantém a continuidade semântica entre os fragmentos.

Para ver mais sobre _"text splitting"_, recomendo a leitura do artigo ['Text Splitters in LangChain for Data Processing'](https://medium.com/data-and-beyond/text-splitters-in-langchain-for-data-processing-3a958eea2797) no Medium.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# quebra conteudo do documento em pedacos
chunks_txt = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(text_document)
type(chunks_txt)

list

In [4]:
len(chunks_txt)

5

In [5]:
chunks_txt

[Document(metadata={'source': '../documents/GTB_gold_Nov23.txt'}, page_content='1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  para  a \nMas

## Gera vetores dos pedaços de texto e armazena em banco de dados vetorial

Agora que temos os fragmentos de texto (_chunks_), vamos gerar a representação numérica deles através do processo de indexação vetorial, ou seja, vamos tranformá-los em vetores. 

Nessa etapa, a primeira coisa a ser feita é a definição da estratégia usada para converter texto em vetores. Essa função fica por conta da classe [`OpenAIEmbeddings`](https://reference.langchain.com/python/langchain-openai/embeddings/base/OpenAIEmbeddings), que percorrerá os `chunks` e, para cada um deles, vai gerar um vetor numérico (lista de coordenadas que descreve o significado daquele trecho específico). Essa classe faz a conexão com a API da OpenAI para realizar a transformação de texto em vetores, funcionando como um modelo de tradução: recebe um texto (ou uma lista de textos) e retorna uma lista de números (o vetor de alta dimensão) que representa o significado semântico desse conteúdo, utilizando um modelo pré-treinado da OpenAI para criar essas representações numéricas.

Agora que temos os vetores, precisamos armazená-los em um "banco de dados vetorial" (_vector store_), uma estrutura de dados otimizada para busca por similiaridade de vetores, que permita comparar esses vetores rapidamente para encontrar quais possuem significados parecidos. Faremos isso com o [FAISS](https://sj-langchain.readthedocs.io/en/latest/vectorstores/langchain.vectorstores.faiss.FAISS.html) (_Facebook AI Similarity Search_), uma biblioteca desenvolvida pela [Meta](https://engineering.fb.com/2017/03/29/data-infrastructure/Faiss-a-library-for-efficient-similarity-search/), projetada especificamente para realizar buscas de similaridade de forma extremamente rápida e eficiente em grandes conjuntos de vetores.

O [FAISS](https://langchain-opentutorial.gitbook.io/langchain-opentutorial/09-vectorstore/11-faiss) constrói um índice otimizado, uma espécie de "mapa" onde cada vetor está posicionado em um espaço multidimensional (baseado em busaca por "vizinhos próximos"), permitindo que a pergunta seja comparada apenas com os _chunks_ mais similares. Quando realizamos uma pergunta, o FAISS compara o vetor da nossa pergunta com os vetores armazenados, calculando a distância matemática entre eles. Os fragmentos com menor distância (maior similaridade) são recuperados.

Primeiro faremos o FAISS receber os _chunks_ e o modelo de _embeddings_ para construir o mapa do espaço vetorial. Então, através do método `as_retriever`, transformaremos o banco de dados em um objeto que segue o padrão de [`Retriever`](https://docs.langchain.com/oss/python/deepagents/retrieval) do LangChain, o que vai permitir que ele seja facilmente integrado em cadeias de execução com o operador `|` (_pipe_).

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# gera vetores dos pedacos de texto
embeddings    = OpenAIEmbeddings() # nao eh necessario passar a chave da API, o langchain faz uma busca por uma variavel de ambiente chamada OPENAI_API_KEY
retriever_txt = FAISS.from_documents(chunks_txt, embeddings).as_retriever(search_kwargs={"k": 2})
type(retriever_txt)

langchain_core.vectorstores.base.VectorStoreRetriever

In [7]:
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x752049ec0c50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x752047332510>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [ ]:
for i in retriever_txt:
    print(i)

('name', None)
('tags', ['FAISS', 'OpenAIEmbeddings'])
('metadata', None)
('vectorstore', <langchain_community.vectorstores.faiss.FAISS object at 0x752049eec500>)
('search_type', 'similarity')
('search_kwargs', {'k': 2})


Quando temos um objeto FAISS (ou qualquer outro _vector store_) e chamamos o método `as_retriever`, criamos uma interface usada pelo LangChain para pesquisar os _chunks_, funcionando essencialmente como uma ponte entre a pergunta e os dados armazenados. Ele tranforma o banco de dados vetorial de uma estrutura estática de armazenamento em um componente funcional de busca. Ao invés de "manualmente" transformar a pergunta em vetor, comparar com o banco e extrair os resultados, o [`Retriever`](https://docs.langchain.com/oss/python/integrations/retrievers) faz isso por debaixo dos panos.

Além disso, note que, ao usar `search_kwargs={"k":2}`, estamos definindo a "estratégia" de busca, instruindo esse componente a sempre retornar os 2 documentos (ou pedaços) com a maior similaridade matemática em relação à consulta feita.

Resumindo, o `as_retriever` prepara o banco vetorial para ser o "fornecedor de contexto" da IA generativa. Ele pega a pergunta que entra na cadeia, busca os _chunks_ relevantes no FAISS e os entrega formatados para serem incluídos no _prompt_.

## Implementa _prompt_ com contexto adicional

E finalmente, podemos montar o _prompt template_:

In [9]:
from langchain_core.prompts import ChatPromptTemplate

# define template do prompt para pergunta sobre conteudo dos documentos
prompt_insurance = ChatPromptTemplate.from_messages([
    ("system", "Responda usando exclusivamente o conteúdo fornecido como contexto"),
    ("human", "{query}\n\nContexto: \n{context}\n\nResposta:")
])

In [10]:
from langchain_core.output_parsers import StrOutputParser

# define cadeia para pergunta com resposta baseada nos documentos
chain = prompt_insurance | llm | StrOutputParser()

In [ ]:
selected_chunks = retriever_txt.invoke("Como devo proceder caso tenha um item roubado?")
type(selected_chunks)

list

In [12]:
selected_chunks

[Document(id='6097b304-2f26-4fc7-a25b-87513a1f9302', metadata={'source': '../documents/GTB_gold_Nov23.txt'}, page_content='uma perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, \nou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português). \n “cartão” refere-se ao cartão Mastercard Gold. \n \n“portador de cartão”, “vo

In [13]:
selected_chunks[0].page_content

'uma perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, \nou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português). \n “cartão” refere-se ao cartão Mastercard Gold. \n \n“portador de cartão”, “você”, e “seu” referem-se a um  portador do cartão Mastercard Gold, que possui'

In [ ]:
from langchain_core.vectorstores import VectorStore

# encapsula em funcao a busca pelos chunks mais similares a pergunta e disparo da execucao da cadeia para responder usuario
def response(query: str, retriever: VectorStore):
    selected_chunks = vector_data_base.invoke(query)
    context         = "\n\n".join(t.page_content for t in selected_chunks)
    
    return chain.invoke({"query": query, "context": context})

In [ ]:
print(response(query="Como devo proceder caso tenha um item roubado?", retriever=retriever_txt))

Para dar entrada em uma ocorrência ou sinistro relacionado a um item roubado, ligue para o Mastercard Global Service pelo número gratuito específico do seu país. Você também pode ligar a cobrar para os Estados Unidos pelo **1-636-722-8881** e solicitar atendimento em português.

O contexto fornecido não informa outros procedimentos ou documentos necessários.


## Recupera conteúdo de múltiplos documentos externos - Arquivos `.pdf`

Agora vamos considerar a situação em que queremos carregar múltiplos arquivos PDF para nossa base de conhecimento do assistente. Podemos começar importanto a classe [`PyPDFLoader`](https://reference.langchain.com/python/langchain-community/document_loaders/pdf/PyPDFLoader) e carregando o conteúdo de um único arquivo PDF:

In [16]:
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    "../documents/GTB_standard_Nov23.pdf",
    "../documents/GTB_gold_Nov23.pdf",
    "../documents/GTB_platinum_Nov23.pdf"
]

pdf_content = PyPDFLoader(pdf_files[0]).load()
pdf_content

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T09:57:01-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:31:05Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '157657e5-4814-45d2-84a2-a8a6e5e8db4b', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T09:57:01-03:00', 'source': '../documents/GTB_standard_Nov23.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \nPrograma de Cartão da Edição MasterCard Standard \n \nInformações i

In [17]:
len(pdf_content)

6

Note que foi retornada uma lista com 6 objetos, no caso, um para cada página do arquivo PDF que foi carregado. Também é possível carregar todo o conteúdo do arquivo em um único objeto, sem separar por páginas. E então, após isso, podemos carregar todos os arquivos PDF disponíveis e somar as listas com as "páginas" de cada um deles, para ter todas as páginas disponíveis em um único cotentor.

In [18]:
# note que como temos uma lista de listas, precisamos passar uma lista vazia como valor de inicialização
documents = sum(
    [PyPDFLoader(pdf).load() for pdf in pdf_files], 
    []
)

len(documents)

59

In [19]:
sum(
    [[1,2,3], [4,5,6]]
    , []
)

[1, 2, 3, 4, 5, 6]

Então quebramos o conteúdo dos arquivos em pedaçõs, geramos seus vetores e armazenamos na base de dados vetorial:

In [20]:
# quebra conteudo do documento em pedacos
chunks_pdf = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(documents)
len(chunks_pdf)

201

In [21]:
chunks_pdf[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T09:57:01-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:31:05Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '157657e5-4814-45d2-84a2-a8a6e5e8db4b', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T09:57:01-03:00', 'source': '../documents/GTB_standard_Nov23.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \nPrograma de Cartão da Edição MasterCard Standard \n \nInformações im

Uma alternativa é usar o método [`load_and_split`](https://sj-langchain.readthedocs.io/en/latest/document_loaders/langchain.document_loaders.pdf.PyPDFLoader.html#langchain.document_loaders.pdf.PyPDFLoader.load_and_split) para carregar o conteúdo dos documentos e já realizar o _split_:

In [29]:
chunks_pdf_ = sum(
    [PyPDFLoader(pdf).load_and_split() for pdf in pdf_files], 
    []
)

len(chunks_pdf_)

68

In [ ]:
# tamanho do primeiro chunk, splitado no carregamento com `load_and_split`
len(chunks_pdf_[0].page_content)

1227

In [35]:
# tamanho do primeiro chunk, splitado com `RecursiveCharacterTextSplitter`
len(chunks_pdf[0].page_content)

939

Para evitar gerar novamente os _embeddings_ dos mesmos _chunks_ a cada execução, podemos salvar localmente a instância FAISS depois de criá-la, antes de chamar o método `as_retriever()`. Assim, os _chunks_ são vetorizados apenas uma vez, na criação da base, e nas próximas execuções basta carregar o índice salvo e criar um _retriever_ a partir dele.

In [39]:
# gera vetores dos pedacos de texto
vector_data_base_pdf = FAISS.from_documents(chunks_pdf, embeddings)
vector_data_base_pdf.save_local("../data/faiss_pdf")

In [41]:
vector_data_base_pdf_ = FAISS.load_local("../data/faiss_pdf", embeddings, allow_dangerous_deserialization=True)
retriever_pdf = vector_data_base_pdf_.as_retriever(search_kwargs={"k": 3})

E por fim, podemos testar algumas perguntas:

In [42]:
print(response(query="Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?", retriever=retriever_pdf))

Se o item comprado foi roubado, a Compra Protegida do Mastercard Gold poderá cobrir o ocorrido, desde que:

- O item tenha sido pago integralmente com o cartão Mastercard Gold;
- O roubo tenha acontecido nos primeiros 30 dias após a data da compra indicada na nota fiscal;
- O valor respeite o limite de até US$ 200 por incidente e US$ 400 acumulados por ano.

**Como proceder:**
1. Notifique a seguradora o mais rápido possível após o roubo.
2. Apresente o recibo original que comprove o pagamento integral com o cartão.
3. Forneça, dentro do prazo exigido, uma cópia oficial do boletim ou laudo de ocorrência policial.
4. Esteja preparado para fornecer informações adicionais, se forem solicitadas.

O benefício não exige a emissão de bilhete de seguro, mas a cobertura depende do cumprimento dos requisitos de elegibilidade.


In [ ]:
print(response(query="Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão mastercad platinum?", retriever=retriever_pdf))

Pelo contexto fornecido, a cobertura descrita é para **Mastercard Gold**, não para Platinum. Portanto, não dá para confirmar com essas informações se o seu cartão **Mastercard Platinum** oferece a mesma proteção; verifique o guia de benefícios específico do Platinum ou consulte a seguradora/emissor do cartão.

Se as regras do trecho se aplicarem ao seu caso, para pedir cobertura por roubo você deverá:

1. **Avisar a seguradora o mais rápido possível** após o ocorrido.
2. Apresentar o **recibo original**, comprovando que o item foi pago integralmente com o cartão elegível.
3. Enviar uma **cópia oficial do boletim/laudo de ocorrência policial**, dentro do prazo exigido.
4. O roubo deve ter ocorrido no período de **30 dias a partir da compra**. A cobertura indicada tem limite de até **USD 200 por incidente** e **USD 400 por ano**.

A seguradora poderá solicitar informações adicionais.


In [ ]:
print(response(query="quais os principais bebefícios do cartão platinum?", retriever=retriever_pdf))

Os principais benefícios do Mastercard Platinum, conforme o guia, são:

- **Concierge Mastercard Platinum**.
- **Proteção para veículo alugado** (MasterSeguro de Automóveis).
- **Serviços de assistência em viagem**.
- **MasterAssist Plus**, com benefícios médicos de viagem, como despesas médicas e hospitalares no exterior, traslado médico e de corpo, retorno de menores ou idosos e acompanhante e hospedagem em caso de hospitalização prolongada.

A cobertura do MasterAssist Plus pode incluir o titular e dependentes, conforme as condições do guia e desde que estejam incluídos no Bilhete de Seguro Anual.


In [ ]:
print(response(query="quais os principais bebefícios do cartão gold?", retriever=retriever_pdf))

Os principais benefícios mencionados para o Mastercard Gold são:

- **Compra Protegida:** cobertura contra roubo ou furto qualificado e danos físicos ao bem nos primeiros 30 dias após a compra.
- **Proteção de Preço:** possibilidade de receber a diferença se, em até 30 dias, o mesmo produto elegível for anunciado por um preço menor.

Para ambos, é necessário pagar o valor integral da compra com o cartão e guardar os comprovantes. A disponibilidade e a seleção de benefícios podem variar conforme o emissor; consulte sua instituição financeira para confirmar as condições.


In [ ]:
print(response(query="o cartão platinum tem alguma proteção contra roubos?", retriever=retriever_pdf))

Sim. Segundo o contexto, há proteção contra roubo para compras feitas integralmente com o cartão Mastercard Gold: o bem deve ser roubado ou sofrer furto qualificado nos primeiros 30 dias após a compra. O limite é de até US$ 200 por ocorrência e US$ 400 por ano, e é necessário apresentar boletim de ocorrência.

O material fornecido descreve a cobertura do **Gold**, não informa se o **Platinum** tem essa proteção para compras. Para o Platinum, menciona apenas cobertura de bens roubados de dentro de um veículo alugado, até US$ 303 por ocorrência.
